# 第 04 章：高级工具调用 (Smart Tooling)

> **学习定位**：本章是过渡章。
> 你会同时看到两条路线：
> 1. `@chain + config`：看清参数是谁补进去的；
> 2. `create_agent + context/ToolRuntime`：理解封装成 Agent 后，参数注入该挂在哪一层。

根据讲义，本章你将掌握以下核心能力：
- **Pydantic v2 契约建模**：建立坚不可摧的参数校验层。
- **运行时上下文注入**：利用 `InjectedToolArg` 实现安全权限透传。
- **防御式编程实操**：应对小模型的参数生成幻觉。
- **运行时链路组装**：利用 `@chain` 把运行时配置注入到 tool call，再交给 LangChain 执行。


## 1. 契约化建模：Pydantic 约束
通过 Pydantic 显式定义 `args_schema`。

In [11]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
import json

class SearchArgs(BaseModel):
    query: str = Field(description="搜索关键词")
    limit: int = Field(default=5, description="结果数量", ge=1, le=10)

@tool(args_schema=SearchArgs)
def smart_search(query: str, limit: int = 5):
    """增强型搜索工具。"""
    return f"[执行搜索] 关键词: {query}, 数量上限: {limit}"

## 2. 安全穿透：使用 InjectedToolArg 隐藏运行时参数

In [12]:
from typing import Annotated
from langchain_core.tools import InjectedToolArg

@tool
def secure_delete(item_id: str, user_id: Annotated[str, InjectedToolArg]):
    """
    安全删除指定项目。LLM 严禁生成 user_id，该字段由系统在运行时注入。
    """
    return f"[审计成功] 用户 {user_id} 已发起对项目 {item_id} 的删除操作。"


## 3. 运行时注入：把隐藏参数补进 Tool Call，再交给 LangChain
利用 LangChain 的 `@chain` 读取运行时 `config`，将隐藏参数补进 `tool_calls`，再由 Runnable 路由到对应工具。


In [13]:
from copy import deepcopy
from langchain_core.runnables import chain, RunnableLambda
from langchain_core.messages import AIMessage

print("=== 提供给模型的 Tool Call Schema ===")
print(json.dumps(secure_delete.tool_call_schema.model_json_schema(), ensure_ascii=False, indent=2))

@chain
def inject_user_id(ai_msg: AIMessage, config: dict):
    """
    运行时注入器：从 RunnableConfig 读取 user_id，补齐到 tool call 参数中。
    """
    actual_user_id = config.get("configurable", {}).get("user_id", "guest_001")

    tool_calls = []
    for tool_call in ai_msg.tool_calls:
        new_call = deepcopy(tool_call)
        new_call["args"]["user_id"] = actual_user_id
        tool_calls.append(new_call)

    return tool_calls

print("运行时注入器 (Injector) 已就绪。")


=== 提供给模型的 Tool Call Schema ===
{
  "description": "安全删除指定项目。LLM 严禁生成 user_id，该字段由系统在运行时注入。",
  "properties": {
    "item_id": {
      "title": "Item Id",
      "type": "string"
    }
  },
  "required": [
    "item_id"
  ],
  "title": "secure_delete",
  "type": "object"
}
运行时注入器 (Injector) 已就绪。


## 4. 实战：智能文件权限管理系统
通过 `inject_user_id | tool_router.map()` 这条纯 LangChain 链路，把动态上下文交给 Runnable 组合执行。


In [14]:
ACL = {
    "admin_9527": ["readme.md", "secret.txt"],
    "guest_001": ["readme.md"]
}

@tool
def file_operator(file_name: str, action: str, user_id: Annotated[str, InjectedToolArg]):
    """文件操作器。支持 action 为 'read' 或 'delete'。"""
    print(f"--- 正在审计分析: 用户 {user_id} 发起的 {action} 请求 ---")
    allowed_files = ACL.get(user_id, [])
    if file_name not in allowed_files:
        return f"错误：用户 {user_id} 对 {file_name} 权限不足。"
    return f"成功：已完成对 {file_name} 的 {action} 操作。"

tool_map = {"file_operator": file_operator}
tool_router = RunnableLambda(
    lambda tool_call: tool_map[tool_call["name"]].invoke(tool_call["args"])
)

secure_file_chain = inject_user_id | tool_router.map()

# 模拟 AI 生成的意向：读取 secret.txt（注意：args 里没有 user_id）
mock_intent = AIMessage(
    content="",
    tool_calls=[{"name": "file_operator", "args": {"file_name": "secret.txt", "action": "read"}, "id": "lab_id", "type": "tool_call"}]
)

print("--- 场景 A: Guest 访客视角 ---")
res_guest = secure_file_chain.invoke(mock_intent, config={"configurable": {"user_id": "guest_001"}})
print(f"结果: {res_guest[0]}")

print("\n--- 场景 B: Admin 管理员视角 ---")
res_admin = secure_file_chain.invoke(mock_intent, config={"configurable": {"user_id": "admin_9527"}})
print(f"结果: {res_admin[0]}")


--- 场景 A: Guest 访客视角 ---
--- 正在审计分析: 用户 guest_001 发起的 read 请求 ---
结果: 错误：用户 guest_001 对 secret.txt 权限不足。

--- 场景 B: Admin 管理员视角 ---
--- 正在审计分析: 用户 admin_9527 发起的 read 请求 ---
结果: 成功：已完成对 secret.txt 的 read 操作。


In [15]:
# 追加示例：把第 01 章的 llm 接进来，给下方 create_agent + astream(..., context=...) 示例复用
# 运行前请先确保当前内核里已经有 llm（可先执行 01_Getting_Started.ipynb 中的模型初始化单元）

import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 统一实例化：2026 年推荐做法
# 如果 api.deepseek.com 无法访问，可以尝试修改 base_url 或使用代理
llm = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    base_url="https://api.deepseek.com",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    streaming=True
)

print(f"模型加载成功：{llm.__class__.__name__}")

模型加载成功：ChatDeepSeek


In [18]:
# 追加示例：沿用第 01 章 create_agent + astream 的写法，通过 ToolRuntime 从 agent context 读取 user_id
from dataclasses import dataclass
from langchain.agents import create_agent
from langgraph.prebuilt import ToolRuntime

@dataclass
class FileRuntimeContext:
    user_id: str

@tool
def file_operator_agent(file_name: str, action: str, runtime: ToolRuntime):
    """文件操作器。支持 action 为 'read' 或 'delete'。"""
    user_id = runtime.context.user_id
    print(f"\n[ToolRuntime] 从 agent context 读取 user_id = {user_id}")
    print(f"--- 正在审计分析: 用户 {user_id} 发起的 {action} 请求 ---")

    allowed_files = ACL.get(user_id, [])
    if file_name not in allowed_files:
        return f"错误：用户 {user_id} 对 {file_name} 权限不足。"
    return f"成功：已完成对 {file_name} 的 {action} 操作。"

file_agent = create_agent(
    model=llm,
    tools=[file_operator_agent],
    context_schema=FileRuntimeContext,
    system_prompt="你是一个文件权限助手。凡是读取或删除文件，都必须调用 file_operator_agent 工具。action 只能是 read 或 delete，不要直接编造结果。",
)

def prepare_file_inputs(user_query: str, chat_history: list = None):
    history = chat_history or []
    return {
        "messages": [
            *history,
            ("user", user_query),
        ]
    }

async def run_file_agent_demo(query: str, user_id: str):
    input_dict = prepare_file_inputs(query)
    full_response = None

    print(f"\n--- 提问: {query} ---")
    print(f"--- 运行时 user_id: {user_id} ---\n")

    async for chunk, metadata in file_agent.astream(
        input_dict,
        context=FileRuntimeContext(user_id=user_id),
        stream_mode="messages",
        version="v2",
    ):
        node = metadata.get("langgraph_node")

        if node == "model" and chunk.content:
            print(chunk.content, end="", flush=True)

        if node == "model":
            full_response = chunk if full_response is None else full_response + chunk

        if node == "tools" and getattr(chunk, "content", None):
            print(f"\n[工具结果] {chunk.content}\n")

    return full_response

# 运行前请先确保当前内核里已经有 llm（可先执行 01_Getting_Started.ipynb 中的模型初始化单元）
final_msg = await run_file_agent_demo("请读取 secret.txt", "admin_9527")
print(f"\n\n[聚合完毕] 最终回答: {final_msg.content}")



--- 提问: 请读取 secret.txt ---
--- 运行时 user_id: admin_9527 ---

我需要读取 secret.txt 文件，让我使用文件操作器来执行这个操作。

/Users/cyrus/work/my/langchain-logbook/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=FileRuntimeContext(user_id='admin_9527'), input_type=FileRuntimeContext])
  return self.__pydantic_serializer__.to_python(



[ToolRuntime] 从 agent context 读取 user_id = admin_9527
--- 正在审计分析: 用户 admin_9527 发起的 read 请求 ---

[工具结果] 成功：已完成对 secret.txt 的 read 操作。

我已经成功读取了 secret.txt 文件。文件读取操作已完成。

[聚合完毕] 最终回答: 我需要读取 secret.txt 文件，让我使用文件操作器来执行这个操作。我已经成功读取了 secret.txt 文件。文件读取操作已完成。
